# 와인 리뷰를 저장하고 추천 근거로 사용하기 — 선택 종합

와인 리뷰 CSV를 문서로 바꾸어 Chroma에 저장하고, 음식 설명과 관련된 리뷰를 검색합니다. 검색 원문을 근거로 모델이 제안한 와인 후보를 대조합니다.


### 준비. CSV와 전용 저장 경로

- **CSV**: 열 이름과 데이터 행으로 구성한 표 형식의 파일.
- **`MAX_REVIEWS`**: 파일 앞에서 읽을 최대 데이터 행 수. 검색 결과 개수와 구분.
- **전용 저장소**: 정책 실습과 다른 디스크 경로·컬렉션을 사용해 문서 묶음 분리.


In [1]:
from pathlib import Path
from dotenv import load_dotenv
import csv
from itertools import islice
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()
CSV_PATH = Path.cwd() / "data" / "wine_reviews_sample.csv"
DATABASE_DIR = Path.cwd() / ".chroma-lab" / "wine-reviews-200-v1"


## 1. CSV에서 리뷰 문서 준비

### 1-1. CSV의 한 행 확인

- **`csv.DictReader`**: 열 이름을 키로 사용해 데이터 행을 딕셔너리로 읽는 도구.
- **`next(reader)`**: 다음 데이터 행 하나를 꺼냄. 열 이름 행 자체와 구분.
- **파일의 `with` 블록**: 블록 종료 시 파일을 닫음. 읽어 둔 문자열·딕셔너리는 변수에 남음.


In [3]:
with CSV_PATH.open(encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    columns = reader.fieldnames
    sample_row = next(reader)
print(columns)
print(type(sample_row))

['title', 'description']
<class 'dict'>


### 1-2. 리뷰를 Document로 바꾸는 함수

- **변환 기준**: description은 본문, title·행 번호는 metadata에 저장.
- **`islice(reader, limit)`**: 읽을 데이터 행의 범위를 제한하는 반복자.
- **`enumerate(..., start=1)`**: 행 딕셔너리와 표시 번호를 함께 제공.
- **반환값**: 제목·본문이 모두 있는 행을 모은 `list[Document]`.


In [4]:
def load_reviews(csv_path, limit=200):
    documents = []
    with csv_path.open(encoding="utf-8-sig", newline="") as file:
        for number, row in enumerate(islice(csv.DictReader(file), limit), start=1):
            title, review = row["title"].strip(), row["description"].strip()
            if title and review:
                documents.append(Document(
                    page_content=review,
                    metadata={"title": title, "review_number": number}
                ))
    return documents

### 1-3. 변환 결과 관찰

- **자료형 변화**: CSV 행 딕셔너리 → `Document` 객체 → 문서 객체 리스트.
- **내용 구분**: `metadata`에서 와인 제목·리뷰 번호, `page_content`에서 영어 원문 확인.


In [5]:
documents = load_reviews(CSV_PATH)
print(type(documents), len(documents))
print(documents[0].metadata)
print(documents[0].page_content)

<class 'list'> 200
{'title': 'Nicosia 2013 Vulkà Bianco  (Etna)', 'review_number': 1}
Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.


## 2. 리뷰 벡터 저장과 검색

### 2-1. 와인 저장소 준비

- **저장 연결**: 와인 전용 컬렉션과 디스크 경로에 임베딩 객체 연결.
- **객체 준비**: Chroma 연결 생성과 리뷰 본문 임베딩·추가 작업은 별도 호출.


### 2-2. 리뷰 최초 저장 또는 재사용

- **최초 추가**: 리뷰 본문 임베딩과 ID·metadata 저장.
- **리뷰 ID**: 데이터 행 번호를 사용한 `review-번호` 형식의 식별자.
- **재사용 범위**: 저장 자료가 있으면 추가 생략. 변경된 CSV와 기존 저장 내용을 자동 비교하지 않음.


### 2-3. 음식 설명으로 리뷰 검색하고 원문 확인

- **검색 입력**: `FOOD`에 담긴 음식 설명 문자열.
- **`k=3`**: 저장된 리뷰에서 가져올 최대 문서 수. CSV를 읽은 200행 제한과 구분.
- **검색 반환값**: 질문 임베딩과 유사도 비교로 얻은 `list[Document]`.
- **원문과 생성문**: 여기서 읽는 것은 영어 리뷰 원문. 추천 답변 생성은 다음 단계.


## 3. 근거를 넣어 추천 요청

### 3-1. 검색 원문을 요청으로 묶기

- **근거 구성**: 검색 Document의 번호·제목·원문을 리뷰별 문자열로 변환.
- **`join()`**: 문자열 목록을 빈 줄로 이어 하나의 context 문자열 생성.
- **`<context>` 태그**: 근거 자료의 경계를 표시하는 텍스트. 지시를 차단하는 실행 기능은 아님.
- **반환 타입**: 음식·조건·근거가 담긴 요청 문자열 하나.


### 3-2. 이번 음식과 근거가 든 요청

- **현재 요청**: 검색한 원문을 포함해 모델에게 보낼 문자열.
- **근거 없음 처리**: 결과가 비면 모델 호출 전에 중단해 근거 없는 추천을 방지.
- **출처 확인**: 리뷰 번호·제목·영어 표현을 답변과 대조할 기준으로 사용.


### 3-3. 추천 모델 준비

- **`ChatOpenAI`**: 리뷰 근거를 읽고 답변을 생성할 채팅 모델 연결.
- **역할 차이**: `OpenAIEmbeddings`는 검색용 벡터 생성, 채팅 모델은 추천 본문 생성.


### 3-4. 추천 답변과 리뷰 근거 대조

- **모델 입력**: 음식과 검색 근거를 함께 담은 prompt 문자열.
- **모델 반환값**: `AIMessage` 객체. `.content`로 추천 본문 읽기.
- **대조 기준**: 제안한 와인·리뷰 번호·영어 표현이 검색 원문과 연결되는지 확인.
